In [ ]:
# import importlib
import sqlite3

from unga80.config import DATA_DIR
from unga80 import data_collection as dc
from unga80 import database as dbase

db = DATA_DIR / "countries.db"

In [ ]:
# Create table `country` if not exists
# dbase.drop_country()
dbase.create_db()

In [ ]:
# Download spreasheet with the urls to the speeches
df = dc.get_video_url_list()

In [ ]:
# Download the transcripts using the speeches urls
dc.download_all_transcripts(df, overwrite=False)

In [ ]:
# Insert initial data
for i, r in df.iterrows():
    start, end = dc.get_seconds_from_str(r["start"], r["end"])
    corpus = dc.get_corpus_from_file(
        country=r["country"], start=start, end=end, path=DATA_DIR / "2025"
    )
    if corpus:
        dbase.insert(country=r["country"], url=r["url"], speech=corpus)

In [ ]:
dbase.select_country()

In [ ]:
query = """UPDATE countries
        SET iso_3=?
        WHERE country=?;"""
for i, r in df.iterrows():
    # print(r["country"], r["iso_3"])
    with sqlite3.connect(db) as conn:
        cursor = conn.cursor()
        cursor.execute(
            query,
            (
                r["iso_3"],
                r["country"],
            ),
        )
        conn.commit()
        cursor.close()